### Отдельная MLForecast-модель на каждый `resist_id` — `17_model_comparison_local_mlforecast_excel_by_resist`

Копия конвейера [`16_model_comparison_local_mlforecast`](16_model_comparison_local_mlforecast.ipynb): **`MLForecast.fit`** по каждому **`resist_id`**, **wide**-данные, **`op_*`** с **`add_op_shift_lags`**, **`calc_*`** с **`add_calc_shift_lags`** / **`add_calc_history_flatten`**.

**Отличие в Excel:** после листов **«Итог»**, **«По resist_id»**, **«MLForecast_params»** и **«Лучшие»** для **каждого** `resist_id` записывается лист **`{resist_id} Данные теста`** — те же колонки, что в [`14_model_comparison_multi_calc_resist_warnings`](14_model_comparison_multi_calc_resist_warnings.ipynb): **`fact_value (факт)`**, **`fact_predict (текущий)`**, **`{Конфигурация} | {Набор признаков} (новый)`** по строкам теста.

Файл: [`model_comparison_per_resist_mlforecast_test_detail.xlsx`](../data/processed/model_comparison_per_resist_mlforecast_test_detail.xlsx).

**On-premise (on-premise):** без внешних API. Тест: июль–декабрь 2025.


In [ ]:
from __future__ import annotations

import json
import sys
import time
import copy
from pathlib import Path

import importlib

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import scripts.calc_lag_features as _calc_lag_mod
import scripts.calc_history_flatten as _calc_hist_mod
import scripts.json_nested_flatten as _json_flat_mod

importlib.reload(_calc_lag_mod)
importlib.reload(_calc_hist_mod)
importlib.reload(_json_flat_mod)

from scripts.calc_lag_features import (
    add_calc_shift_lags,
    add_op_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
    DEFAULT_OP_SHIFTS,
    scalar_calc_columns,
)
from scripts.calc_history_flatten import add_calc_history_flatten
from scripts.json_nested_flatten import flatten_nested_dict

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, ExponentiallyWeightedMean

%matplotlib inline
import warnings

warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)


#### Конфигурация

In [ ]:
RESIST_ID = 124
RESIST_IDS = [124]
GROUP_COLS = ("resist_id", "operation_id")
SUMMARY_TOP = 15

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]
HORIZON = len(TEST_MONTHS)

CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_employFact_7",
    "calc_resist_10",
]

OUT_XLSX = Path("../data/processed/model_comparison_per_resist_mlforecast_test_detail.xlsx")

# Для global (legacy) можно было использовать resist_code + op_code; для одной модели на resist_id — только op_code
STATIC_FEATURES_PER_RESIST = ["op_code"]
STATIC_FEATURES = ["resist_code", "op_code"]  # не используется в MLForecast после перехода на per-resist
# Служебные колонки wide: не экзогенные для MLForecast (id, время, таргет, коды)
SERVICE_COLS = {
    "resist_id",
    "operation_id",
    "year",
    "month",
    "date",
    "fact_value",
    "fact_predict",
    "op_code",
    "resist_code",
}


def _dyn_exog_columns(wide: pd.DataFrame, extra_exclude=None) -> list[str]:
    ex = set(SERVICE_COLS) | (extra_exclude or set())
    return [c for c in wide.columns if c not in ex]


#### Загрузка данных: полный JSON `calculate_parameter` / `operation_parameter` (активные операции)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_nested_dict(calc_obj, prefix="calc")
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        op_flat = flatten_nested_dict(payload, prefix="op")
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
        }
        records.append({**base_record, **calc_flat, **op_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")

for c in df_model39.columns:
    if c in ("operation_id", "resist_id"):
        continue
    if df_model39[c].dtype == object:
        num = pd.to_numeric(df_model39[c], errors="coerce")
        if num.notna().sum() >= 0.5 * len(df_model39):
            df_model39[c] = num.fillna(0.0)
        else:
            df_model39[c] = pd.factorize(df_model39[c].astype(str))[0].astype(np.float64)
    else:
        df_model39[c] = pd.to_numeric(df_model39[c], errors="coerce").fillna(0.0)

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
op_cols = [c for c in df_model39.columns if c.startswith("op_")]
print(f"calc_* колонок: {len(calc_cols)}, op_* колонок: {len(op_cols)}")

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Метрики

In [ ]:
from typing import Union


def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0,
            100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


def _clip_predictions(s: Union[pd.Series, np.ndarray]) -> Union[pd.Series, np.ndarray]:
    """MLForecast может передать в callback и Series, и ndarray."""
    if isinstance(s, pd.Series):
        return pd.Series(
            np.maximum(s.to_numpy(dtype=float), 0.0),
            index=s.index,
            name=s.name,
        )
    return np.maximum(np.asarray(s, dtype=float), 0.0)


#### MLForecast: long format, `fit` на train, рекурсивный `predict(h)` на тесте

In [ ]:
def _exog_series_to_float64(s: pd.Series) -> pd.Series:
    """MLForecast принимает в X только int/float/bool; bool/object из JSON приводим к float64."""
    if pd.api.types.is_bool_dtype(s) or str(s.dtype) == "boolean":
        return s.astype(np.float64)
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(np.float64)
    s2 = s.replace(
        {
            True: 1.0,
            False: 0.0,
            "True": 1.0,
            "False": 0.0,
            "true": 1.0,
            "false": 0.0,
        }
    )
    out = pd.to_numeric(s2, errors="coerce")
    if out.isna().all() and len(s2) > 0:
        out = pd.Series(pd.factorize(s2.astype(str))[0], index=s2.index, dtype=np.float64)
    return out.fillna(0.0).astype(np.float64)


def _before_predict_fill_nan(df: pd.DataFrame) -> pd.DataFrame:
    """Sklearn GradientBoostingRegressor (GBR) не принимает NaN/inf в X; callback (callback) MLForecast до predict."""
    if not isinstance(df, pd.DataFrame):
        return df
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = np.nan_to_num(
            out[c].to_numpy(dtype=float, copy=True),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
    return out


def _wide_to_long(
    wide: pd.DataFrame,
    dynamic_cols: list[str],
    *,
    static_cols: list[str],
    use_operation_as_unique_id: bool = False,
) -> pd.DataFrame:
    w = wide.copy()
    w["ds"] = pd.to_datetime(dict(year=w["year"], month=w["month"], day=1), errors="coerce")
    if use_operation_as_unique_id:
        w["unique_id"] = w["operation_id"].astype(str)
    else:
        w["unique_id"] = w["resist_id"].astype(str) + "__" + w["operation_id"].astype(str)
    w["y"] = w["fact_value"]
    use = ["unique_id", "ds", "y"] + static_cols + dynamic_cols
    for c in dynamic_cols:
        w[c] = _exog_series_to_float64(w[c])
    for c in static_cols:
        w[c] = _exog_series_to_float64(w[c])
    return w[use].dropna(subset=["ds", "y"]).sort_values(["unique_id", "ds"]).reset_index(drop=True)


def _build_mlf_models() -> dict[str, object]:
    return {
        "LGBM-default": LGBMRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            verbose=-1,
            random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=10,
            verbose=-1,
            random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            verbosity=0,
            random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800,
            max_depth=8,
            learning_rate=0.05,
            reg_alpha=0.1,
            reg_lambda=5,
            verbosity=0,
            random_state=42,
        ),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "DT-ccp": DecisionTreeRegressor(
            max_depth=15,
            min_samples_leaf=2,
            ccp_alpha=0.02,
            random_state=42,
        ),
        "RF-default": RandomForestRegressor(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        "GBR-default": GradientBoostingRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
        ),
    }


# Без add_fact_lag_columns: лаги целевого ряда задаёт MLForecast
df = df_model39[df_model39["resist_id"].isin(RESIST_IDS)].copy()
df = df.sort_values(list(GROUP_COLS) + ["year", "month"], kind="mergesort").reset_index(drop=True)
df["op_code"] = df["operation_id"].astype("category").cat.codes
df["resist_code"] = df["resist_id"].astype("category").cat.codes

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)

df, _op_lag_cols, _op_keys_used = add_op_shift_lags(
    df,
    drop_scalar_op=True,
    shifts=DEFAULT_OP_SHIFTS,
    group_cols=GROUP_COLS,
)
print(
    f"op_* лаги (operation_parameter, M-1): ключей {len(_op_keys_used)}, "
    f"колонок {len(_op_lag_cols)}; сдвиги: {DEFAULT_OP_SHIFTS}"
)

_num_calc = [
    c
    for c in scalar_calc_columns(df)
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]
df_shift, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(
    df.copy(),
    calc_keys=_num_calc,
    drop_scalar_calc=True,
    group_cols=GROUP_COLS,
)
b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)
print(
    f"calc_* lag-сдвиги: ключей {len(calc_keys_used)}; "
    f"набор B: {len(b_flat)}, набор C: {len(c_flat)}; shift-месяцы: {DEFAULT_CALC_SHIFTS}"
)

df_flat, flatten_calc_cols, calc_keys_hist = add_calc_history_flatten(
    df.copy(), drop_scalar_calc=True, merge_keys=GROUP_COLS
)
b_hist_flat = flatten_calc_cols
print(
    f"full history flatten: ключей calc {len(calc_keys_hist)}, "
    f"колонок calc_k__YYYY_MM {len(flatten_calc_cols)}"
)

_c_only_drop = set(b_flat) - set(c_flat)
FEATURE_SETS: dict[str, list[str]] = {
    "B+lags": _dyn_exog_columns(df_shift),
    "C+lags": _dyn_exog_columns(df_shift, extra_exclude=_c_only_drop),
    "B_hist+lags": _dyn_exog_columns(df_flat),
}
FEATURE_DF = {
    "B+lags": df_shift,
    "C+lags": df_shift,
    "B_hist+lags": df_flat,
}

for fs_name, fs_cols in FEATURE_SETS.items():
    dfx = FEATURE_DF[fs_name]
    miss = [c for c in fs_cols if c not in dfx.columns]
    if miss:
        raise ValueError(f"Не хватает колонок для {fs_name}: {miss[:20]}...")
    print(
        f"  {fs_name}: экзогенных колонок {len(fs_cols)} (+ static {STATIC_FEATURES_PER_RESIST}; лаги y — MLForecast)"
    )

mask_test = (df_shift["year"] == TEST_YEAR) & (df_shift["month"].isin(TEST_MONTHS))
mask_train = ~mask_test
n_test = int(mask_test.sum())
n_train = int(mask_train.sum())
print(f"Всего train: {n_train} строк, test: {n_test} строк (по всем RESIST_IDS)")

if n_train == 0:
    print("Предупреждение: глобальный train пуст; по каждому resist_id будут только нулевые метрики.")

LAG_TRANSFORMS = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        ExponentiallyWeightedMean(alpha=0.3),
    ],
}
DATE_FEATURES = ["month", "quarter", "year"]
LAGS_MLF = [1, 2, 3, 6, 12]

results: list = []
preds_store: dict[tuple[str, str, int], np.ndarray] = {}
model_configs = _build_mlf_models()

for rid in RESIST_IDS:
    m = df_shift["resist_id"] == rid
    mask_train_rid = mask_train & m
    mask_test_rid = mask_test & m
    n_train_rid = int(mask_train_rid.sum())
    if mask_test_rid.any():
        t_fp, s_fp, q_fp = calc_metrics(
            df_shift.loc[mask_test_rid, "fact_value"].values,
            df_shift.loc[mask_test_rid, "fact_predict"].values,
        )
    else:
        t_fp, s_fp, q_fp = 0.0, 0.0, 0.0

    if n_train_rid == 0:
        print(f"resist_id={rid}: train пуст — без fit, нулевые метрики для всех конфигураций")
        for fs_name in FEATURE_SETS:
            for config_name in model_configs:
                results.append(
                    {
                        "resist_id": rid,
                        "Конфигурация": config_name,
                        "Набор признаков": fs_name,
                        "Тип": "MLForecast (per resist_id, recursive)",
                        "Точность, %": 0.0,
                        "Стабильность, %": 0.0,
                        "Качество, %": 0.0,
                        "Время, сек": 0.0,
                        "Точн. fact_predict, %": t_fp,
                        "Стаб. fact_predict, %": s_fp,
                        "Кач. fact_predict, %": q_fp,
                    }
                )
        continue

    if not mask_test_rid.any():
        print(f"resist_id={rid}: нет тестовых строк — без predict, нулевые метрики")
        for fs_name in FEATURE_SETS:
            for config_name in model_configs:
                results.append(
                    {
                        "resist_id": rid,
                        "Конфигурация": config_name,
                        "Набор признаков": fs_name,
                        "Тип": "MLForecast (per resist_id, recursive)",
                        "Точность, %": 0.0,
                        "Стабильность, %": 0.0,
                        "Качество, %": 0.0,
                        "Время, сек": 0.0,
                        "Точн. fact_predict, %": t_fp,
                        "Стаб. fact_predict, %": s_fp,
                        "Кач. fact_predict, %": q_fp,
                    }
                )
        continue

    print(
        f"resist_id={rid}: train={n_train_rid}, test={int(mask_test_rid.sum())}; "
        f"baseline fact_predict: Т={t_fp}%, С={s_fp}%, К={q_fp}%"
    )

    for fs_name, dyn_cols in FEATURE_SETS.items():
        wide = FEATURE_DF[fs_name]
        long_train = _wide_to_long(
            wide.loc[mask_train_rid],
            dyn_cols,
            static_cols=STATIC_FEATURES_PER_RESIST,
            use_operation_as_unique_id=True,
        )
        if long_train.empty:
            print(f"  resist_id={rid} | {fs_name}: пустой long train — пропуск")
            for config_name in model_configs:
                results.append(
                    {
                        "resist_id": rid,
                        "Конфигурация": config_name,
                        "Набор признаков": fs_name,
                        "Тип": "MLForecast (per resist_id, recursive)",
                        "Точность, %": 0.0,
                        "Стабильность, %": 0.0,
                        "Качество, %": 0.0,
                        "Время, сек": 0.0,
                        "Точн. fact_predict, %": t_fp,
                        "Стаб. fact_predict, %": s_fp,
                        "Кач. fact_predict, %": q_fp,
                    }
                )
            continue

        _rows_per_uid = long_train.groupby("unique_id").size()
        _max_hist = int(_rows_per_uid.max())
        _max_lag = max(LAGS_MLF)
        # Иначе после лагов (lags) / окон (rolling) матрица X пустая → ValueError: 2 dimensional and non empty
        if _max_hist <= _max_lag:
            print(
                f"  resist_id={rid} | {fs_name}: max длина ряда по операции={_max_hist} <= max(lags)={_max_lag} — пропуск fit"
            )
            for config_name in model_configs:
                results.append(
                    {
                        "resist_id": rid,
                        "Конфигурация": config_name,
                        "Набор признаков": fs_name,
                        "Тип": "MLForecast (per resist_id, recursive)",
                        "Точность, %": 0.0,
                        "Стабильность, %": 0.0,
                        "Качество, %": 0.0,
                        "Время, сек": 0.0,
                        "Точн. fact_predict, %": t_fp,
                        "Стаб. fact_predict, %": s_fp,
                        "Кач. fact_predict, %": q_fp,
                    }
                )
            continue

        sub_te = wide.loc[mask_test_rid].copy()
        exog_test = sub_te[dyn_cols].copy()
        for c in dyn_cols:
            exog_test[c] = _exog_series_to_float64(exog_test[c])
        exog_test["unique_id"] = sub_te["operation_id"].astype(str)
        exog_test["ds"] = pd.to_datetime(
            dict(year=sub_te["year"], month=sub_te["month"], day=1), errors="coerce"
        )

        df_test_meta_rid = wide.loc[mask_test_rid, ["resist_id", "operation_id", "year", "month"]].copy()
        df_test_meta_rid["unique_id"] = df_test_meta_rid["operation_id"].astype(str)
        df_test_meta_rid["ds"] = pd.to_datetime(
            dict(
                year=df_test_meta_rid["year"],
                month=df_test_meta_rid["month"],
                day=1,
            ),
            errors="coerce",
        )
        order_keys = df_test_meta_rid[["unique_id", "ds"]].copy()
        y_test_rid = wide.loc[mask_test_rid, "fact_value"].values

        for config_name, model in model_configs.items():
            fcst = MLForecast(
                models=copy.deepcopy(model),
                freq="MS",
                lags=LAGS_MLF,
                lag_transforms=LAG_TRANSFORMS,
                date_features=DATE_FEATURES,
                num_threads=-1,
            )
            try:
                fcst.fit(long_train, static_features=STATIC_FEATURES_PER_RESIST)
            except ValueError as err:
                _msg = str(err).lower()
                if "non empty" in _msg or "2 dimensional" in _msg:
                    print(
                        f"  resist_id={rid} | {fs_name}: fit невозможен (пустой X после лагов/окон): {err}"
                    )
                    for cn in model_configs:
                        results.append(
                            {
                                "resist_id": rid,
                                "Конфигурация": cn,
                                "Набор признаков": fs_name,
                                "Тип": "MLForecast (per resist_id, recursive)",
                                "Точность, %": 0.0,
                                "Стабильность, %": 0.0,
                                "Качество, %": 0.0,
                                "Время, сек": 0.0,
                                "Точн. fact_predict, %": t_fp,
                                "Стаб. fact_predict, %": s_fp,
                                "Кач. fact_predict, %": q_fp,
                            }
                        )
                    break
                raise
            fd = fcst.make_future_dataframe(h=HORIZON)
            X_df = fd.merge(
                exog_test[["unique_id", "ds"] + dyn_cols],
                on=["unique_id", "ds"],
                how="left",
            )
            for _c in dyn_cols:
                X_df[_c] = _exog_series_to_float64(X_df[_c])
            X_df[dyn_cols] = X_df[dyn_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            t0 = time.perf_counter()
            preds = fcst.predict(
                h=HORIZON,
                X_df=X_df,
                before_predict_callback=_before_predict_fill_nan,
                after_predict_callback=_clip_predictions,
            )
            elapsed = round(time.perf_counter() - t0, 2)
            pred_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
            merged = order_keys.merge(
                preds[["unique_id", "ds", pred_col]],
                on=["unique_id", "ds"],
                how="left",
            )
            y_pred = np.maximum(np.nan_to_num(merged[pred_col].to_numpy(dtype=float), nan=0.0), 0.0)
            combo_key = (config_name, fs_name, rid)
            preds_store[combo_key] = y_pred
            tochnost, stab, kachestvo = calc_metrics(y_test_rid, y_pred)
            results.append(
                {
                    "resist_id": rid,
                    "Конфигурация": config_name,
                    "Набор признаков": fs_name,
                    "Тип": "MLForecast (per resist_id, recursive)",
                    "Точность, %": tochnost,
                    "Стабильность, %": stab,
                    "Качество, %": kachestvo,
                    "Время, сек": elapsed,
                    "Точн. fact_predict, %": t_fp,
                    "Стаб. fact_predict, %": s_fp,
                    "Кач. fact_predict, %": q_fp,
                }
            )
            print(f"  {config_name} | {fs_name}: Качество={kachestvo}% ({elapsed}s)")

df_results = pd.DataFrame(results)
if not df_results.empty:
    df_results = df_results.sort_values(["Качество, %", "resist_id"], ascending=[False, True]).reset_index(
        drop=True
    )
print("\n--- Топ по Качеству ---")
if df_results.empty:
    print("(нет строк результатов)")
else:
    print(df_results.head(SUMMARY_TOP).to_string(index=False))

df_per_rid = df_results.copy()
if not df_per_rid.empty:
    df_per_rid = df_per_rid.sort_values(["resist_id", "Качество, %"], ascending=[True, False]).reset_index(
        drop=True
    )

MLF_PARAMS = {
    "mode": "per_resist_id",
    "freq": "MS",
    "lags": LAGS_MLF,
    "lag_transforms": str(LAG_TRANSFORMS),
    "date_features": DATE_FEATURES,
    "horizon": HORIZON,
}
df_params = pd.DataFrame([MLF_PARAMS])

OUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df_results.to_excel(w, sheet_name="Итог", index=False)
    df_per_rid.to_excel(w, sheet_name="По resist_id", index=False)
    df_params.to_excel(w, sheet_name="MLForecast_params", index=False)
print(f"\nСохранено: {OUT_XLSX.resolve()}")


In [ ]:
# Лист «Лучшие» + по каждому resist_id лист «{id} Данные теста» (формат как в notebook 14)


def _best_combo_label(br: pd.Series) -> str:
    fs = br["Набор признаков"]
    cfg = br["Конфигурация"]
    if fs is None or (isinstance(fs, float) and pd.isna(fs)):
        return str(cfg)
    fs_s = str(fs).strip()
    if fs_s in ("", "—"):
        return str(cfg)
    return f"{cfg} | {fs_s}"


df_best = pd.DataFrame()
if not df_results.empty:
    df_best = (
        df_results.sort_values(
            ["Качество, %", "Конфигурация", "Набор признаков"],
            ascending=[False, True, True],
        )
        .drop_duplicates("resist_id", keep="first")
        .sort_values("resist_id")
        .reset_index(drop=True)
    )

per_rid_test_detail: dict[int, pd.DataFrame] = {}

for _, br in df_best.iterrows():
    rid = int(br["resist_id"])
    cfg = br["Конфигурация"]
    fs = br["Набор признаков"]
    key = (cfg, fs, rid)
    if key not in preds_store:
        print(f"Нет предсказаний в preds_store для {key}, лист «Данные теста» пропуск")
        continue
    y_hat = preds_store[key]
    m = mask_test & (df_shift["resist_id"] == rid)
    base = df_shift.loc[m, ["resist_id", "operation_id", "year", "month", "fact_value", "fact_predict"]].reset_index(
        drop=True
    )
    if len(base) != len(y_hat):
        print(
            f"resist_id={rid}: несовпадение длин теста ({len(base)}) и прогноза ({len(y_hat)}), пропуск"
        )
        continue
    best_combo = _best_combo_label(br)
    df_one = base.copy()
    df_one["predict_new"] = y_hat
    df_one = df_one[
        ["resist_id", "operation_id", "year", "month", "fact_value", "fact_predict", "predict_new"]
    ]
    df_one = df_one.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
    df_one = df_one.rename(
        columns={
            "fact_value": "fact_value (факт)",
            "fact_predict": "fact_predict (текущий)",
            "predict_new": f"{best_combo} (новый)",
        }
    )
    per_rid_test_detail[rid] = df_one

if per_rid_test_detail:
    try:
        from IPython.display import display

        r0 = min(per_rid_test_detail.keys())
        display(per_rid_test_detail[r0].head(20))
    except Exception:
        r0 = min(per_rid_test_detail.keys())
        print(per_rid_test_detail[r0].head(20).to_string())
else:
    print("(нет листов «Данные теста» — пусто)")

if not OUT_XLSX.exists():
    raise FileNotFoundError(
        f"Нет файла {OUT_XLSX}; сначала выполните предыдущую ячейку с экспортом Excel."
    )

with pd.ExcelWriter(
    OUT_XLSX,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as w:
    df_best.to_excel(w, sheet_name="Лучшие", index=False)
    for rid in sorted(per_rid_test_detail.keys()):
        per_rid_test_detail[rid].to_excel(w, sheet_name=f"{rid} Данные теста", index=False)

print(
    f"Лист «Лучшие» и листы «Данные теста» по resist_id записаны: {OUT_XLSX.resolve()}\n"
    f"  resist_id: {sorted(per_rid_test_detail.keys())}"
)
